# 01 — Replaceable proposals and certified row attacks

This notebook constructs a controlled, explicitly non-market seller
policy; trains grouped out-of-fold proposal models; measures proposal
stability; and attacks declared constraint rows using an exact
coordinate-wise box oracle.

The exact oracle certifies the strongest perturbation for each
targeted linear row under the stated box. It does not claim a global
optimum for the sum of all violations. Stress boxes use explicitly
labeled feasible family-boundary centers: an additive center for
decompositions and an equal-price center for derivations. These are
controlled stress constructions, not observed seller prices. Proposal
models remain outside the safety argument.


In [1]:
import subprocess
import sys

# Install only dependencies that are absent or version-sensitive in Colab.
# The final environment manifest records every resolved version.
packages = [
    "duckdb==1.3.2",
    "osqp==1.0.4",
    "sqlglot==27.0.0",
    "pyarrow==19.0.1",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", *packages]
)
print("Required packages are available.")


Required packages are available.


In [2]:
from pathlib import Path
import hashlib
import importlib
import json
import os
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_GOOGLE_DRIVE = True
if IN_COLAB and USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
    default_root = "/content/drive/MyDrive/Arbiter_ICDE_2027"
else:
    default_root = "/content/Arbiter_ICDE_2027"

PROJECT_ROOT = Path(os.environ.get("ARBITER_PROJECT_ROOT", default_root))
active_path = PROJECT_ROOT / "active_run.json"
if not active_path.exists():
    raise FileNotFoundError(
        "Run Notebook 00 first. It creates active_run.json and the shared code."
    )
active = json.loads(active_path.read_text(encoding="utf-8"))
RUN_MODE = active["run_mode"]
if active.get("suite_version") != "2026-08-01-audited-v4":
    raise RuntimeError(
        "The active run was created by an older suite. Run the audited "
        "Notebook 00 before continuing."
    )
RUN_DIR = PROJECT_ROOT / active["run_directory"]
CODE_DIR = PROJECT_ROOT / "code"
if not CODE_DIR.exists():
    raise FileNotFoundError("Shared code is missing. Rerun Notebook 00.")
manifest_path = CODE_DIR / "code_manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError("Shared-code manifest is missing. Rerun Notebook 00.")
expected_hashes = json.loads(manifest_path.read_text(encoding="utf-8"))
for name, expected_hash in expected_hashes.items():
    module_path = CODE_DIR / name
    if not module_path.is_file():
        raise FileNotFoundError(f"Shared module is missing: {name}")
    observed_hash = hashlib.sha256(module_path.read_bytes()).hexdigest()
    if observed_hash != expected_hash:
        raise RuntimeError(f"Shared module hash mismatch: {name}")
code_dir_text = str(CODE_DIR.resolve())
if code_dir_text not in sys.path:
    sys.path.insert(0, code_dir_text)
for module_name in [
    "arbiter_core",
    "arbiter_workload",
    "arbiter_experiments",
    "arbiter_data",
]:
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
config = json.loads((RUN_DIR / "run_config.json").read_text(encoding="utf-8"))
print(f"Active run: {RUN_MODE}")
print(f"Run directory: {RUN_DIR}")


Mounted at /content/drive
Active run: full
Run directory: /content/drive/MyDrive/Arbiter_ICDE_2027/runs/full_audited_v4


In [3]:
import pandas as pd
import numpy as np
import shutil

from arbiter_core import (
    build_snapshot,
    independent_verify,
    run_enforcement_method,
)
from arbiter_experiments import (
    add_structural_features,
    grouped_oof_predictions,
    make_feasible_controlled_policy,
    run_certified_attack_campaign,
    run_omitted_relation_gap_campaign,
)

workload_dir = RUN_DIR / "00_workload"
output_dir = RUN_DIR / "01_proposals_attacks"
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
all_assets = pd.read_csv(workload_dir / "sql_assets.csv")
assets = all_assets[
    all_assets["expected_supported"]
    .astype(str)
    .str.lower()
    .eq("true")
].copy()
relations = pd.read_csv(workload_dir / "compiled_relations.csv")
features = add_structural_features(assets)

policy_rows = []
policy_meta_rows = []
for dataset, dataset_assets in assets.groupby("dataset", sort=False):
    ordered_ids = dataset_assets["asset_id"].astype(str).tolist()
    snapshot = build_snapshot(
        ordered_ids,
        relations[relations["dataset"] == dataset],
        epoch=1,
    )
    ordered_features = (
        features[features["dataset"] == dataset]
        .set_index("asset_id")
        .loc[ordered_ids]
        .reset_index()
    )
    prices, metadata = make_feasible_controlled_policy(
        snapshot,
        ordered_features,
        tolerance=config["tolerance"],
    )
    check = independent_verify(
        snapshot, prices, tolerance=config["tolerance"]
    )
    for asset_id, price in zip(ordered_ids, prices, strict=True):
        policy_rows.append(
            {
                "dataset": dataset,
                "asset_id": asset_id,
                "target_price": float(price),
                "price_semantics": (
                    "controlled experimental seller-policy units"
                ),
            }
        )
    policy_meta_rows.append(
        {
            "dataset": dataset,
            "assets": snapshot.assets,
            "constraints": snapshot.constraints,
            "nonzeros": snapshot.nonzeros,
            "verified": check.passed,
            "max_residual": check.max_residual,
            **metadata,
        }
    )

policy = pd.DataFrame(policy_rows)
policy_metadata = pd.DataFrame(policy_meta_rows)
policy.to_csv(output_dir / "controlled_policy.csv", index=False)
policy_metadata.to_csv(
    output_dir / "controlled_policy_metadata.csv", index=False
)
display(policy_metadata)


,dataset,assets,constraints,nonzeros,verified,max_residual,projected,solver,l1_adjustment,verified_max_residual
0,NYC Taxi 2024-01,4096,13543,25233,True,3.993252e-17,False,none,0.0,3.993252e-17
1,TPC-H SF1,4096,13296,24556,True,2.081668e-17,False,none,0.0,2.081668e-17
2,UCI Adult,4096,13946,26160,True,5.421011e-20,False,none,0.0,5.421011e-20


In [4]:
fold_metrics, oof_predictions, pooled_metrics = (
    grouped_oof_predictions(
        features,
        policy,
        seeds=config["seeds"],
        model_names=[
            "hist_gradient_boosting",
            "mlp",
            "dummy_median",
        ],
        folds=5,
    )
)
fold_metrics.to_csv(
    output_dir / "model_metrics_grouped_folds.csv", index=False
)
oof_predictions.to_csv(
    output_dir / "oof_predictions.csv", index=False
)
pooled_metrics.to_csv(
    output_dir / "model_metrics_pooled.csv", index=False
)
display(
    pooled_metrics.groupby(["dataset", "model"])
    .agg(
        r2_min=("pooled_r2", "min"),
        r2_median=("pooled_r2", "median"),
        r2_max=("pooled_r2", "max"),
        mae_median=("pooled_mae", "median"),
    )
    .reset_index()
)


,dataset,model,r2_min,r2_median,r2_max,mae_median
0,NYC Taxi 2024-01,dummy_median,-0.475657,-0.475657,-0.475657,0.359922
1,NYC Taxi 2024-01,hist_gradient_boosting,0.992028,0.992028,0.992028,0.004347
2,NYC Taxi 2024-01,mlp,0.999521,0.999687,0.999837,0.006044
3,TPC-H SF1,dummy_median,-0.419761,-0.419761,-0.419761,0.349141
4,TPC-H SF1,hist_gradient_boosting,0.995808,0.995808,0.995808,0.002113
5,TPC-H SF1,mlp,0.996647,0.999039,0.999480,0.008760
6,UCI Adult,dummy_median,-0.465910,-0.465910,-0.465910,0.367933
7,UCI Adult,hist_gradient_boosting,0.991863,0.991863,0.991863,0.004151
8,UCI Adult,mlp,0.999386,0.999716,0.999771,0.006063


In [5]:
# Natural proposal audit: useful for liveness, not adversarial stress.
natural_rows = []
for (
    dataset,
    model,
    seed,
), group in oof_predictions.groupby(["dataset", "model", "seed"]):
    dataset_assets = assets[assets["dataset"] == dataset]
    ordered_ids = dataset_assets["asset_id"].astype(str).tolist()
    snapshot = build_snapshot(
        ordered_ids,
        relations[relations["dataset"] == dataset],
        epoch=1,
    )
    proposal_map = group.set_index("asset_id")[
        "oof_prediction"
    ].to_dict()
    proposal = np.asarray(
        [proposal_map[asset_id] for asset_id in ordered_ids],
        dtype=float,
    )
    before = independent_verify(
        snapshot, proposal, tolerance=config["tolerance"]
    )
    result = run_enforcement_method(
        snapshot,
        proposal,
        method="arbiter_active",
        tolerance=config["tolerance"],
        max_sweeps=config["repair_sweeps"],
        active_row_budget=config["repair_active_row_budget"],
        step=config["repair_step"],
        osqp_max_iter=config["osqp_max_iter"],
    )
    natural_rows.append(
        {
            "dataset": dataset,
            "model": model,
            "seed": int(seed),
            "violations_before": before.violations,
            "max_residual_before": before.max_residual,
            "violations_after": result.verification.violations,
            "max_residual_after": result.verification.max_residual,
            "released": not result.rejected,
            "fallback_used": result.fallback_used,
            "unsafe_release": bool(
                not result.rejected
                and not result.verification.passed
            ),
            "gate_ms": result.runtime_ms,
            "l1_adjustment": result.l1_adjustment,
        }
    )
natural_gate = pd.DataFrame(natural_rows)
natural_gate.to_csv(
    output_dir / "natural_proposal_gate.csv", index=False
)
display(natural_gate)


,dataset,model,seed,violations_before,max_residual_before,violations_after,max_residual_after,released,fallback_used,unsafe_release,gate_ms,l1_adjustment
0,NYC Taxi 2024-01,dummy_median,11,893,1.199190e-06,0,9.837108e-08,True,False,False,16.476018,0.000575
1,NYC Taxi 2024-01,dummy_median,23,893,1.199190e-06,0,9.837108e-08,True,False,False,16.495018,0.000575
2,NYC Taxi 2024-01,dummy_median,37,893,1.199190e-06,0,9.837108e-08,True,False,False,16.872619,0.000575
3,NYC Taxi 2024-01,dummy_median,51,893,1.199190e-06,0,9.837108e-08,True,False,False,28.982292,0.000575
4,NYC Taxi 2024-01,dummy_median,73,893,1.199190e-06,0,9.837108e-08,True,False,False,30.625614,0.000575
5,NYC Taxi 2024-01,hist_gradient_boosting,11,2219,4.613465e-01,0,9.991128e-08,True,False,False,57.283934,3.043821
6,NYC Taxi 2024-01,hist_gradient_boosting,23,2219,4.613465e-01,0,9.991128e-08,True,False,False,58.461260,3.043821
7,NYC Taxi 2024-01,hist_gradient_boosting,37,2219,4.613465e-01,0,9.991128e-08,True,False,False,59.641269,3.043821
8,NYC Taxi 2024-01,hist_gradient_boosting,51,2219,4.613465e-01,0,9.991128e-08,True,False,False,60.805148,3.043821
9,NYC Taxi 2024-01,hist_gradient_boosting,73,2219,4.613465e-01,0,9.991128e-08,True,False,False,61.909162,3.043821


In [6]:
attacks, attack_tokens, sparse_deltas = (
    run_certified_attack_campaign(
        assets=assets,
        relations=relations,
        policy=policy,
        budgets=config["attack_budgets"],
        requested_cases_per_relation_type=config[
            "attack_cases_per_relation_type"
        ],
        minimum_cases_per_relation_type=config[
            "minimum_attack_cases_per_relation_type"
        ],
        tolerance=config["tolerance"],
        repair_options={
            "max_sweeps": config["repair_sweeps"],
            "active_row_budget": config[
                "repair_active_row_budget"
            ],
            "step": config["repair_step"],
            "osqp_max_iter": config["osqp_max_iter"],
        },
    )
)
attacks.to_csv(
    output_dir / "certified_row_attacks.csv", index=False
)
attack_tokens.to_csv(
    output_dir / "attack_verification_records.csv", index=False
)
sparse_deltas.to_csv(
    output_dir / "attack_sparse_price_deltas.csv", index=False
)
attack_summary = (
    attacks.groupby(["dataset", "budget_fraction"], as_index=False)
    .agg(
        attacks=("attack_id", "size"),
        distinct_target_rows=("target_row_id", "nunique"),
        attack_success_rate=(
            "violations_before",
            lambda values: float((values > 0).mean()),
        ),
        released_rate=("released", "mean"),
        fallback_rate=("fallback_used", "mean"),
        unsafe_release_rate=("unsafe_release", "mean"),
        gate_p50_ms=("repair_ms", "median"),
        gate_p95_ms=(
            "repair_ms",
            lambda values: float(np.quantile(values, 0.95)),
        ),
        l1_median=("l1_adjustment", "median"),
        max_post_residual=("max_residual_after", "max"),
    )
)
attack_summary["target_reuse_rate"] = 1.0 - (
    attack_summary["distinct_target_rows"]
    / attack_summary["attacks"]
)
attack_summary.to_csv(
    output_dir / "certified_row_attack_summary.csv", index=False
)
attack_family_summary = (
    attacks.groupby(
        [
            "dataset",
            "budget_fraction",
            "target_constraint_type",
        ],
        as_index=False,
    )
    .agg(
        attacks=("attack_id", "size"),
        distinct_target_rows=("target_row_id", "nunique"),
        attack_success_rate=(
            "violations_before",
            lambda values: float((values > 0).mean()),
        ),
        released_rate=("released", "mean"),
        fallback_rate=("fallback_used", "mean"),
        unsafe_release_rate=("unsafe_release", "mean"),
        gate_p50_ms=("repair_ms", "median"),
        gate_p95_ms=(
            "repair_ms",
            lambda values: float(np.quantile(values, 0.95)),
        ),
        l1_median=("l1_adjustment", "median"),
    )
)
attack_family_summary["target_reuse_rate"] = 1.0 - (
    attack_family_summary["distinct_target_rows"]
    / attack_family_summary["attacks"]
)
attack_family_summary.to_csv(
    output_dir
    / "certified_row_attack_family_summary.csv",
    index=False,
)
display(attack_summary)
display(attack_family_summary)


,dataset,budget_fraction,attacks,distinct_target_rows,attack_success_rate,released_rate,fallback_rate,unsafe_release_rate,gate_p50_ms,gate_p95_ms,l1_median,max_post_residual,target_reuse_rate
0,NYC Taxi 2024-01,0.10,240,240,1.0,1.0,0.0,0.0,1.145256,1.364277,0.000002,3.993252e-17,0.0
1,NYC Taxi 2024-01,0.25,240,240,1.0,1.0,0.0,0.0,0.825636,1.306134,0.000005,3.993252e-17,0.0
2,NYC Taxi 2024-01,0.50,240,240,1.0,1.0,0.0,0.0,0.685876,0.966237,0.000010,3.993252e-17,0.0
3,TPC-H SF1,0.10,240,240,1.0,1.0,0.0,0.0,0.676724,0.763661,0.007748,2.081668e-17,0.0
4,TPC-H SF1,0.25,240,240,1.0,1.0,0.0,0.0,0.676263,0.769998,0.019369,2.081668e-17,0.0
5,TPC-H SF1,0.50,240,240,1.0,1.0,0.0,0.0,0.671464,0.811552,0.038738,2.081668e-17,0.0
6,UCI Adult,0.10,240,240,1.0,1.0,0.0,0.0,1.196223,1.464585,0.000008,5.421011e-20,0.0
7,UCI Adult,0.25,240,240,1.0,1.0,0.0,0.0,0.741842,1.228988,0.000019,5.421011e-20,0.0
8,UCI Adult,0.50,240,240,1.0,1.0,0.0,0.0,0.726485,1.080223,0.000038,5.421011e-20,0.0


,dataset,budget_fraction,target_constraint_type,attacks,distinct_target_rows,attack_success_rate,released_rate,fallback_rate,unsafe_release_rate,gate_p50_ms,gate_p95_ms,l1_median,target_reuse_rate
0,NYC Taxi 2024-01,0.10,decomposition,80,80,1.0,1.0,0.0,0.0,1.144142,1.280819,0.272000,0.0
1,NYC Taxi 2024-01,0.10,filter_derivation,80,80,1.0,1.0,0.0,0.0,1.214425,1.387559,0.000002,0.0
2,NYC Taxi 2024-01,0.10,projection_derivation,80,80,1.0,1.0,0.0,0.0,1.057859,1.192593,0.000002,0.0
3,NYC Taxi 2024-01,0.25,decomposition,80,80,1.0,1.0,0.0,0.0,1.156319,1.721570,0.680000,0.0
4,NYC Taxi 2024-01,0.25,filter_derivation,80,80,1.0,1.0,0.0,0.0,1.027610,1.281073,0.000005,0.0
5,NYC Taxi 2024-01,0.25,projection_derivation,80,80,1.0,1.0,0.0,0.0,0.641205,0.756120,0.000005,0.0
6,NYC Taxi 2024-01,0.50,decomposition,80,80,1.0,1.0,0.0,0.0,0.672057,0.770410,1.360000,0.0
7,NYC Taxi 2024-01,0.50,filter_derivation,80,80,1.0,1.0,0.0,0.0,0.728088,1.001931,0.000010,0.0
8,NYC Taxi 2024-01,0.50,projection_derivation,80,80,1.0,1.0,0.0,0.0,0.650679,0.992205,0.000010,0.0
9,TPC-H SF1,0.10,decomposition,80,80,1.0,1.0,0.0,0.0,0.667029,0.701807,0.272000,0.0


In [7]:
# Coverage-boundary rerun with auditable relation identifiers.
# Each case omits exactly one supported relation row, attacks that
# row, and then applies the ordinary gate to the incomplete snapshot.
gap_cases = run_omitted_relation_gap_campaign(
    assets=assets,
    relations=relations,
    policy=policy,
    budgets=config["attack_budgets"],
    cases_per_relation_type=(
        3 if config["run_mode"] == "pilot" else 20
    ),
    tolerance=config["tolerance"],
    repair_options={
        "max_sweeps": config["repair_sweeps"],
        "active_row_budget": config[
            "repair_active_row_budget"
        ],
        "step": config["repair_step"],
        "osqp_max_iter": config["osqp_max_iter"],
    },
)
gap_summary = (
    gap_cases.groupby(
        ["dataset", "budget_fraction", "relation_type"],
        as_index=False,
    )
    .agg(
        cases=("relation_id", "size"),
        distinct_relations=("relation_id", "nunique"),
        candidate_target_violation_rate=(
            "certified_target_residual",
            lambda values: float(
                (values > config["tolerance"]).mean()
            ),
        ),
        incomplete_release_rate=(
            "released_by_incomplete_gate",
            "mean",
        ),
        gap_survival_rate=("gap_survives", "mean"),
        fallback_rate=("fallback_used", "mean"),
        gate_p50_ms=("gate_ms", "median"),
    )
)
gap_cases.to_csv(
    output_dir / "omitted_relation_gap_cases.csv",
    index=False,
)
gap_summary.to_csv(
    output_dir / "omitted_relation_gap_summary.csv",
    index=False,
)
display(gap_summary)


,dataset,budget_fraction,relation_type,cases,distinct_relations,candidate_target_violation_rate,incomplete_release_rate,gap_survival_rate,fallback_rate,gate_p50_ms
0,NYC Taxi 2024-01,0.10,decomposition,20,20,1.0,1.0,1.00,0.0,0.918100
1,NYC Taxi 2024-01,0.10,filter_derivation,20,20,1.0,1.0,0.20,0.0,1.200899
2,NYC Taxi 2024-01,0.10,projection_derivation,20,20,1.0,1.0,1.00,0.0,1.517725
3,NYC Taxi 2024-01,0.25,decomposition,20,20,1.0,1.0,1.00,0.0,1.348639
4,NYC Taxi 2024-01,0.25,filter_derivation,20,20,1.0,1.0,0.10,0.0,1.283379
5,NYC Taxi 2024-01,0.25,projection_derivation,20,20,1.0,1.0,1.00,0.0,1.370731
6,NYC Taxi 2024-01,0.50,decomposition,20,20,1.0,1.0,1.00,0.0,3.235059
7,NYC Taxi 2024-01,0.50,filter_derivation,20,20,1.0,1.0,0.10,0.0,1.918689
8,NYC Taxi 2024-01,0.50,projection_derivation,20,20,1.0,1.0,1.00,0.0,1.479107
9,TPC-H SF1,0.10,decomposition,20,20,1.0,1.0,1.00,0.0,0.910756


In [8]:
expected_attack_cells = {
    (str(dataset), float(budget), str(relation_type))
    for dataset in assets["dataset"].unique()
    for budget in config["attack_budgets"]
    for relation_type in relations.loc[
        relations["dataset"] == dataset, "relation_type"
    ].unique()
}
observed_attack_cells = set(
    attacks[
        [
            "dataset",
            "budget_fraction",
            "target_constraint_type",
        ]
    ].itertuples(index=False, name=None)
)
expected_gap_cells = expected_attack_cells
observed_gap_cells = set(
    gap_cases[
        ["dataset", "budget_fraction", "relation_type"]
    ].itertuples(index=False, name=None)
)
attack_targets_unique = bool(
    not attacks.duplicated(
        [
            "dataset",
            "budget_fraction",
            "target_constraint_type",
            "target_row_id",
        ]
    ).any()
)
gap_targets_unique = bool(
    not gap_cases.duplicated(
        [
            "dataset",
            "budget_fraction",
            "relation_type",
            "relation_id",
        ]
    ).any()
)
expected_center_policy = {
    "decomposition": "additive_boundary_center",
    "projection_derivation": (
        "equal_price_derivation_boundary_center"
    ),
    "filter_derivation": (
        "equal_price_derivation_boundary_center"
    ),
}
center_policy_ok = bool(
    all(
        row.attack_center_policy
        == expected_center_policy[row.target_constraint_type]
        for row in attacks[
            [
                "target_constraint_type",
                "attack_center_policy",
            ]
        ]
        .drop_duplicates()
        .itertuples(index=False)
    )
)
quality = pd.DataFrame(
    [
        {
            "check": "controlled policies verify",
            "severity": "fatal",
            "passed": bool(policy_metadata["verified"].all()),
        },
        {
            "check": "certified campaign contains actual violations",
            "severity": "warning",
            "passed": bool(
                (
                    attack_family_summary[
                        "attack_success_rate"
                    ]
                    >= 0.80
                ).all()
            ),
        },
        {
            "check": "every relation family is attacked in every cell",
            "severity": "fatal",
            "passed": observed_attack_cells == expected_attack_cells,
        },
        {
            "check": "attack target rows are unique within each cell",
            "severity": "fatal",
            "passed": attack_targets_unique,
        },
        {
            "check": "stress-center policy is explicit per relation family",
            "severity": "fatal",
            "passed": center_policy_ok,
        },
        {
            "check": "omitted-row grid is complete",
            "severity": "fatal",
            "passed": observed_gap_cells == expected_gap_cells,
        },
        {
            "check": "omitted target rows are unique within each cell",
            "severity": "fatal",
            "passed": gap_targets_unique,
        },
        {
            "check": "no measured unsafe release",
            "severity": "fatal",
            "passed": bool(
                attack_summary["unsafe_release_rate"].max() == 0
            ),
        },
        {
            "check": "all released vectors pass post-verification",
            "severity": "fatal",
            "passed": bool(
                attacks.loc[
                    attacks["released"], "violations_after"
                ].max()
                == 0
            ),
        },
        {
            "check": "one OOF prediction per asset/model/seed",
            "severity": "fatal",
            "passed": bool(
                not oof_predictions.duplicated(
                    ["dataset", "model", "seed", "asset_id"]
                ).any()
            ),
        },
        {
            "check": "all OOF predictions are finite",
            "severity": "fatal",
            "passed": bool(
                np.isfinite(
                    oof_predictions["oof_prediction"].to_numpy(float)
                ).all()
            ),
        },
        {
            "check": "MLP has nonnegative pooled R2 in every cell",
            "severity": "warning",
            "passed": bool(
                (
                    pooled_metrics.loc[
                        pooled_metrics["model"] == "mlp",
                        "pooled_r2",
                    ]
                    >= 0
                ).all()
            ),
        },
        {
            "check": "gap cases retain relation and digest identity",
            "severity": "fatal",
            "passed": bool(
                len(gap_cases)
                and gap_cases["relation_id"].notna().all()
                and gap_cases[
                    "complete_constraint_digest"
                ].notna().all()
                and gap_cases[
                    "incomplete_constraint_digest"
                ].notna().all()
                and (
                    gap_cases["complete_constraint_digest"]
                    != gap_cases["incomplete_constraint_digest"]
                ).all()
            ),
        },
        {
            "check": (
                "incomplete-gate releases pass every encoded row"
            ),
            "severity": "fatal",
            "passed": bool(
                (
                    gap_cases.loc[
                        gap_cases[
                            "released_by_incomplete_gate"
                        ],
                        "incomplete_violations_after",
                    ]
                    == 0
                ).all()
            ),
        },
    ]
)
quality.to_csv(output_dir / "stage_quality_checks.csv", index=False)
display(quality)
fatal_failures = quality[
    (quality["severity"] == "fatal") & (~quality["passed"])
]
if len(fatal_failures):
    raise RuntimeError(
        "Notebook 01 has a fatal quality failure. Do not use its "
        "results in the manuscript."
    )
if not quality["passed"].all():
    print(
        "Notebook 01 completed with a warning. The evidence package "
        "will mark the weak claim unsupported."
    )
else:
    print("Notebook 01 PASS. Continue to Notebook 02.")


,check,severity,passed
0,controlled policies verify,fatal,True
1,certified campaign contains actual violations,warning,True
2,every relation family is attacked in every cell,fatal,True
3,attack target rows are unique within each cell,fatal,True
4,stress-center policy is explicit per relation ...,fatal,True
5,omitted-row grid is complete,fatal,True
6,omitted target rows are unique within each cell,fatal,True
7,no measured unsafe release,fatal,True
8,all released vectors pass post-verification,fatal,True
9,one OOF prediction per asset/model/seed,fatal,True


Notebook 01 PASS. Continue to Notebook 02.
